# Retriever, LCEL 체인의 표준 인터페이스
- 이전에는 Chroma vectorDB 에 문서를 저장하고 `similarity_search()` 로 직접 검색했음
- 이번에는 대한민국 헌법 PDF를 로드해 vectorstore 를 만들고 **Retriever** 인터페이스로 검색 전략을 바꿔 봄
- Retriever 는 `invoke(질문) -> list[Document]` 형태의 표준 검색 인터페이스


## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [ ]:
# 필요한 라이브러리 설치
# uv add -qU langchain langchain-chroma langchain-openai langchain-text-splitters python-dotenv chromadb pypdf


## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

## 2. Retriever 핵심 개념

| 개념 | 설명 |
|---|---|
| Loader | PDF, 웹, 텍스트 파일을 `Document` 목록으로 읽어옴 |
| Splitter | 긴 문서를 검색 가능한 작은 청크로 나눔 |
| VectorStore | 청크를 임베딩해 저장하고 직접 검색하는 저장소 |
| Retriever | 질문을 받아 관련 문서 목록을 반환하는 표준 인터페이스 |
| `search_kwargs` | `k`, `filter`, `score_threshold` 같은 검색 옵션 |
| `search_type` | `similarity`, `mmr`, `similarity_score_threshold` |
| LCEL 연결 | `retriever | format_docs` 처럼 체인 안에 삽입 |


## 3. 헌법 PDF 로드 + 청크 분할

- `PyPDFLoader(..., mode="page")` 는 PDF를 페이지 단위 `Document` 목록으로 읽음
- 각 문서의 metadata 에는 보통 `source`, `page` 같은 출처 정보가 들어감


In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 사용자가 요청한 경로 형식: ../data/...
pdf_path = Path(r"../data/대한민국헌법(헌법)(제00010호)(19880225).pdf")

loader = PyPDFLoader(str(pdf_path), mode="page")
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
docs = text_splitter.split_documents(pages)

doc_ids = [f"constitution-{i:04d}" for i in range(len(docs))]

print(f"PDF 페이지 수: {len(pages)}")
print(f"검색용 청크 수: {len(docs)}")
print("첫 번째 청크 metadata:", docs[0].metadata)
print(docs[0].page_content[:300])

## 4. Chroma vectorstore 만들기

- 헌법 청크를 OpenAI 임베딩으로 벡터화해 Chroma에 저장
- 수업 중 같은 셀을 반복 실행해도 ID가 중복되지 않도록 기존 collection을 삭제한 뒤 다시 만듦


In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
collection_name = "constitution_retriever"

# 반복 실행 대비: 같은 collection이 있으면 지우고 다시 생성
reset_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
)
reset_store.delete_collection()

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    ids=doc_ids,
    collection_name=collection_name,
)

print("저장 문서 수:", vectorstore._collection.count())


## 5. 기본 retriever, `as_retriever()`
- `similarity_search()` 는 vectorstore의 메서드
- `as_retriever()` 를 사용하면 LCEL 체인에 넣기 쉬운 표준 retriever가 됨


In [ ]:
retriever =

results =
for doc in results:
    print(f"[page={doc.metadata.get('page')}] {doc.page_content[:180]}\n")


## 6. 메타데이터 필터 retriever

- PDF 로더가 붙인 `page` metadata 를 이용해 특정 페이지 범위 안에서만 검색할 수 있음
- 페이지 번호는 0부터 시작


In [ ]:
page_retriever =

results =
for doc in results:
    print(f"[page={doc.metadata.get('page')}] {doc.page_content[:180]}\n")


In [ ]:
source_retriever =

results =
for doc in results:
    print(f"[page={doc.metadata.get('page')}] {doc.page_content[:180]}\n")


## 7. MMR, 중복을 줄이는 다양성 검색

- MMR(Maximal Marginal Relevance)은 질문과 유사하면서도 서로 다른 문서를 골라줌
- 같은 표현의 청크가 반복될 때 유용

In [ ]:
mmr_retriever =

results =
for doc in results:
    print(f"[page={doc.metadata.get('page')}] {doc.page_content[:180]}\n")


## 8. Score threshold, 관련 없는 문서 줄이기

- 자료에 없는 질문인데도 억지로 top-k 문서를 넣으면 LLM이 그럴듯하게 답을 만들 수 있음
- 질문과 너무 동떨어진 청크를 LLM에 넣지 않도록 임계값 설정

> `similarity_score_threshold` 의 `score_threshold` 는 LangChain relevance score 기준입니다. 높을수록 더 엄격합니다.


In [ ]:
threshold_retriever =

question1 =

results =

print(f"\n질문: {question1}")
print(f"가져온 문서 수: {len(results)}")
for doc in results:
    print(f"  [page={doc.metadata.get('page')}] {doc.page_content[:120]}")

In [ ]:
question2 =

results2 =

print(f"\n질문: {question2}")
print(f"가져온 문서 수: {len(results2)}")
for doc in results2:
    print(f"  [page={doc.metadata.get('page')}] {doc.page_content[:120]}")

## 9. 검색 결과 포맷팅

- Retriever 는 `Document` 목록을 반환합니다. LLM 프롬프트에는 출처와 본문이 잘 보이도록 문자열로 바꿔 넣습니다.


In [ ]:
from langchain_core.documents import Document

def format_docs(docs: list[Document]) -> str:
    if not docs:
        return "검색된 참고 자료가 없습니다."

    formatted = []
    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata
        formatted.append(
            f"[{i}] source={meta.get('source')}, page={meta.get('page')}\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(formatted)


sample_docs = retriever.invoke(" ")
print(format_docs(sample_docs))


## 10. LCEL RAG 체인에 retriever 연결


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

llm =

RAG_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",

    ),
    ("user", "참고 자료:\n{context}\n\n질문: {question}"),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }

)


## 11. 실행해보기


## 12. 정리

- `PyPDFLoader(..., mode="page")` 로 PDF를 페이지 단위 `Document`로 읽을 수 있습니다.
- `RecursiveCharacterTextSplitter` 로 페이지 문서를 검색용 청크로 나눕니다.
- `vectorstore.as_retriever()` 는 LCEL 체인에 넣기 좋은 표준 검색 인터페이스입니다.
- metadata 필터는 PDF의 `source`, `page` 같은 출처 정보로 제한할 수 있습니다.
- MMR 은 유사도와 다양성을 같이 고려해 중복 청크를 줄입니다.
- threshold retriever 는 관련도가 낮은 문서를 프롬프트에 넣지 않아 환각을 줄이는 데 도움됩니다.


## [실습]

1. 기본 retriever 의 `k` 값을 1, 3, 5 로 바꿔 `국회의원의 의무` 검색 결과를 비교합니다.
2. `filter={"page": {"$lte": 3}}` 조건으로 헌법 앞부분 질문만 검색합니다.
3. MMR 의 `lambda_mult` 를 0, 0.5, 1 로 바꿔 `국민의 권리와 의무` 결과 다양성을 비교합니다.
4. `score_threshold` 를 0.2, 0.35, 0.5 로 바꿔 자료 밖 질문의 반환 문서 수를 확인합니다.
5. `rag_chain` 에 기본 retriever, MMR retriever, threshold retriever 를 각각 연결해 답변 차이를 비교합니다.
